<a href="https://colab.research.google.com/github/KaustubhSN12/Implement-Spark-Streaming_BDA/blob/main/Implement_Spark_Streaming_Big_Data_Analytics_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Libreary
from pyspark import SparkContext
from pyspark.streaming import StreamingContext
import threading
import time
import socket
from sklearn import datasets

In [ ]:
# Step 1: Create Spark Context
sc = SparkContext("local[2]", "InbuiltDatasetStreaming")
#
ssc = StreamingContext(sc, 3)



/usr/local/lib/python3.12/dist-packages/pyspark/streaming/context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(


In [ ]:
# ---------------------------
# Step 2: Streaming Source (Socket)
# ---------------------------
lines = ssc.socketTextStream("localhost", 9999)

In [ ]:
# # ---------------------------
# # Step 3: Processing Logic
# # ---------------------------
# words = lines.flatMap(lambda line: line.split(" "))
# pairs = words.map(lambda word: (word, 1))
# word_counts = pairs.reduceByKey(lambda a, b: a + b)
# # Print results on console
# word_counts.pprint()

In [ ]:
# ---------------------------
# Step 3: Processing Logic
# ---------------------------
words = lines.flatMap(lambda line: line.split(" "))
pairs = words.map(lambda word: (word, 1))
word_counts = pairs.reduceByKey(lambda a, b: a + b)



# Print results on console
word_counts.pprint()

##✅Save results to text files (each micro-batch saved separately)

Output directory: "stream_output"
Spark will create files like "stream_output-/part-0000"
---

word_counts.saveAsTextFiles("stream_output/wordcount")

In [ ]:

# Step 4: Start Streaming

def start_streaming():
    ssc.start()
    ssc.awaitTermination(60)
    ssc.stop(stopSparkContext=True, stopGraceFully=True)

In [ ]:

# Step 5: Simulate Stream

def send_data():
    from sklearn.datasets import fetch_20newsgroups
    # Load inbuilt dataset (subset for demo)
    dataset = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))


    data = dataset.data[:50]

    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    server.bind(("localhost", 9999))
    server.listen(1)
    print("Socket server started on port 9999...")
    conn, addr = server.accept()

    for doc in data:
        # Send only first line of each document
        line = doc.split("\n")[0]
        if line.strip():
            conn.send((line + "\n").encode("utf-8"))
            time.sleep(2)
    conn.close()
    server.close()

In [ ]:
# ---------------------------
# Step 6: Run
# ---------------------------
threading.Thread(target=send_data).start()
start_streaming()


-------------------------------------------
Time: 2025-10-05 06:58:36
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:58:36
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:58:39
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:58:39
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:58:42
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:58:42
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:58:45
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:58:45
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:58:48
----------

Exception in thread Thread-6 (send_data):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-4204854071.py", line 24, in send_data
BrokenPipeError: [Errno 32] Broken pipe


-------------------------------------------
Time: 2025-10-05 06:59:36
-------------------------------------------
('teenage', 1)
('spotty', 1)
('chin', 1)
('and', 1)
('greasy', 1)
('nose.', 1)
('', 1)
('I', 1)
('My', 1)
('14-y-o', 1)
...

-------------------------------------------
Time: 2025-10-05 06:59:39
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:59:39
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:59:42
-------------------------------------------

-------------------------------------------
Time: 2025-10-05 06:59:42
-------------------------------------------

